In [1]:
import requests
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

In [2]:
schedule_2026_url = "https://www.espn.com/mens-college-basketball/team/schedule/_/id/172/season/2026"+'&_xhr=1'
roster_url = "https://www.espn.com/mens-college-basketball/team/roster/_/id/172"+'&_xhr=1'

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

### Get roster/schedule (for 2026,2025,2024)

In [4]:
schedule_json = requests.get(schedule_2026_url, headers=headers)
with open('cache/mens/schedule_2026.json', 'w') as f:
    json.dump(schedule_json.json(), f)
    
roster_json = requests.get(roster_url, headers=headers)
with open('cache/mens/roster.json', 'w') as f:
    json.dump(roster_json.json(), f)

In [5]:
with open('cache/mens/schedule_2026.json') as f:
    d = json.load(f)
temp = d['page']['content']['scheduleData']['teamSchedule'][0]['events']['post']
game_ids_2026 = [x['result']['link'] for x in temp]

In [6]:
with open('cache/mens/roster.json') as f:
    d = json.load(f)
temp = d['page']['content']['roster']['athletes']
players = pd.json_normalize(temp)
players.to_csv('mens/players.csv',index=False)

### Get shots per game (for 2026 only, for now)

In [7]:
for game_id in tqdm(game_ids_2026):
    parsed_id = game_id[game_id.find('gameId/')+7:game_id.find('gameId/')+16]
    game_json = requests.get(game_id+'&_xhr=1', headers=headers, timeout=10)
    with open('cache/mens/'+parsed_id+'.json', 'w') as f:
        json.dump(game_json.json(), f)

  0%|          | 0/28 [00:00<?, ?it/s]

100%|██████████| 28/28 [00:17<00:00,  1.64it/s]


In [8]:
shots = pd.DataFrame()

for game_id in (game_ids_2026):
    parsed_id = game_id[game_id.find('gameId/')+7:game_id.find('gameId/')+16]
    with open('cache/mens/'+parsed_id+'.json') as f:
        d = json.load(f)
    shots_temp = pd.json_normalize(d['page']['content']['gamepackage']['shtChrt']['plays'])
    shots_temp['gameId'] = parsed_id
    shots = pd.concat([shots, shots_temp], axis=0)

In [9]:
shots.to_csv('mens/shots.csv',index=False)